<a href="https://colab.research.google.com/github/ha199/Medical-Transcription-SOAP-Note-Generation/blob/main/Copy_of_autdio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installing the librray

In [1]:
!pip install openai-whisper transformers torch accelerate sentencepiece
!apt-get install ffmpeg -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 45 not upgraded.


🔹 Cell 2 — Upload Audio

In [2]:
from google.colab import files
uploaded = files.upload()

Saving sample_dictation.mp3 to sample_dictation (2).mp3


🔹 Cell 3 — Whisper Transcription


In [3]:
import whisper

# Load model (same as your previous project)
model = whisper.load_model("base")

# Get uploaded file name
audio_path = list(uploaded.keys())[0]

# Transcribe
result = model.transcribe(audio_path)

transcript = result["text"]

print("TRANSCRIPT:\n")
print(transcript)

# Save
with open("transcript.txt", "w") as f:
    f.write(transcript)

TRANSCRIPT:

 What was you in? I've had this pain on the outside of my right elbow now. I first started noticing it several months ago, but recently it's just been more painful. Okay. So you said several months ago. Did anything happen several months ago? Is there any sort of trigger trauma, anything like that to that area? No, there wasn't any trauma or any triggers that I noticed. I was just feeling it a bit more at the end of work. Yeah, I was just having a feeling of pain a bit more at the end of work. Okay. Does anything make it better or worse, the pain? Yeah, if I really, if I'm just resting the elbow, it makes it better. And I've tried things like ivy profin, which has helped with the pain. I'll do that for helping get through work sometimes if the pain's bad enough. Right. Okay. And if you were to describe the quality of the pain, is it sharp, robby, achy? It's kind of, well, it's achy and then sometimes depending on the movement, it can be sharp as well. It can be sharp. Okay

We can see above the wihper model convert the audio into text i used wishper base model cause Faster wishper was taking time due to capacity


 Cell 4 Installing Groq

In [4]:
!pip install groq

Taking API key as input its just for demo perpose in production i will use keyvault to store the key

In [5]:
from getpass import getpass
import os

os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ API key: ")

Enter your GROQ API key: ··········


 Cell 5 Calling the LLM model to generate response

In [6]:
from groq import Groq

import os
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
prompt = f"""
You are a clinical documentation assistant.

Convert the following medical dictation into a STRICT SOAP note.

IMPORTANT RULES (DO NOT BREAK):
1. Subjective = ONLY what patient says (symptoms, history, feelings)
2. Objective = ONLY doctor observations, exam findings, tests
3. Assessment = ONLY diagnosis or suspected condition
4. Plan = ONLY treatment, medications, follow-up

FORMAT RULES:
- Return ONLY valid JSON
- No extra text before or after
- Keep it concise and structured
- Do NOT mix sections
- Do NOT repeat same sentence in multiple sections
- the subject shoulnd not be more lengthy
- object shoould not contain informations form subject part

- follow these intructions stricktly

EXPECTED FORMAT:

{{
  "Subjective": "",
  "Objective": "",
  "Assessment": "",
  "Plan": ""
}}

Now process this:

{transcript}
"""

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0
)

soap_output = response.choices[0].message.content

print("SOAP OUTPUT:\n")
print(soap_output)

SOAP OUTPUT:

{
  "Subjective": "Patient reports experiencing pain on the outside of their right elbow for several months, with recent increase in pain. Pain is described as achy and sharp, worse with wrist movement and repetitive strain. Patient has tried over-the-counter pain medication and topical ointments for relief. Pain is rated 4/10 and has not prevented daily activities, but is causing issues with work. No other symptoms or medical conditions reported.",
  "Objective": "Physical examination reveals tenderness to palpation on the lateral aspect of the right elbow. Pain is exacerbated by wrist extension and flexion, but not by elbow flexion. No redness, swelling, or deformity noted. Range of motion is full in the elbow and wrist. No numbness, tingling, or coldness in the arm. No other abnormalities noted.",
  "Assessment": "Lateral epicondylitis (tennis elbow) likely diagnosis.",
  "Plan": "Activity modification to avoid repetitive strain on the wrist extensor muscles. Use of co

 🔹 CELL 6 — Extract JSON Cleanly

In [7]:
import json
import re

text = soap_output

match = re.search(r'\{.*\}', text, re.DOTALL)

if match:
    soap_json = json.loads(match.group())
else:
    soap_json = {"error": "JSON not found"}

print("\nSOAP NOTE:\n")
print(json.dumps(soap_json, indent=2))

with open("soap_note.json", "w") as f:
    json.dump(soap_json, f, indent=2)


SOAP NOTE:

{
  "Subjective": "Patient reports experiencing pain on the outside of their right elbow for several months, with recent increase in pain. Pain is described as achy and sharp, worse with wrist movement and repetitive strain. Patient has tried over-the-counter pain medication and topical ointments for relief. Pain is rated 4/10 and has not prevented daily activities, but is causing issues with work. No other symptoms or medical conditions reported.",
  "Objective": "Physical examination reveals tenderness to palpation on the lateral aspect of the right elbow. Pain is exacerbated by wrist extension and flexion, but not by elbow flexion. No redness, swelling, or deformity noted. Range of motion is full in the elbow and wrist. No numbness, tingling, or coldness in the arm. No other abnormalities noted.",
  "Assessment": "Lateral epicondylitis (tennis elbow) likely diagnosis.",
  "Plan": "Activity modification to avoid repetitive strain on the wrist extensor muscles. Use of com

 CELL 7 — Validation if objective has any information from Subject

In [8]:
def validate_soap(soap):
    errors = []

    subjective_keywords = ["reports", "feels", "complains", "states"]

    # Check if subjective phrases leaked into Objective
    for word in subjective_keywords:
        if word in soap.get("Objective", "").lower():
            errors.append("❌ Subjective language found in Objective")

    # Ensure each section exists
    for key in ["Subjective", "Objective", "Assessment", "Plan"]:
        if not soap.get(key):
            errors.append(f"❌ Missing {key}")

    return errors if errors else [" SOAP structure looks valid"]

In [9]:
validation = validate_soap(soap_json)

print("\nVALIDATION:\n")
for v in validation:
    print(v)


VALIDATION:

 SOAP structure looks valid
